# Lecture 21 — subplots, bars, log axes, good fixtures
### EES 3350/5350 · Python in Earth Science · Monday, October 26, 2026 

**Learning objectives**
- make a figure with several axes using `fig, axes = plt.subplots(rows, cols_)`, and pick one axes with `axes[0]` or `axes[0, 1]`
- share an axis between panels with `sharex=True` / `sharey=True`
- make a bar chart with `ax.bar()` for amounts in categories
- switch an axis to a log scale with `ax.set_yscale("log")` and say when that helps
- check a figure against a short list of rules, and fix one that breaks them

---
## Part 0

```
cd ~/course-materials
git pull
cp week09/21_subplots_good_figures.ipynb ~/LastName_EES3350/labs/ees-3350-5350-week09-username/21_subplots_good_figures_LastName.ipynb
```

In [ ]:
!pwd

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

raw = pd.read_csv("../week08/nashville_ghcnd_2025.csv", parse_dates=["DATE"], index_col ="DATE")
#raw = pd.read_csv("../../../course-materials/week08/nashville_ghcnd_2025.csv", parse_dates=["DATE"], index_col ="DATE")

weather = raw[["TMAX", "TMIN", "PRCP", "AWND"]].copy()
weather.head(3)

---
## Part 1 : subplots, more than one graph on a page

Temperature and rain are both in this file, but they have **different units** (F and inches). They don't belong on teh same y-axis, but they do belong on the same **page**, lined up by date, so you can see how they relate.

`plt.subplots()` takes the number of **rows** and **columns** of graphs you want:

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 6))

print(type(axes))
print(axes.shape)

With more than one graph, `plt.subplots()` hands back a NumPy **array** of axes, sow e call it `axes` (plural). Its shape is `(2,)`: two axes, stacked top to bottom. You pick one with indexing, just like any other array:
|**you write**|**you get**|
|:--|:--|
|`axes[0]`| the top graph|
|`axes[1]`| the bottom graph|

Each one is an ordinary `ax`, so everything from earlier works on it: `axes[0].plot()`, `axes[1].set_ylabel()`, and so on.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 6))

# top: temperature
axes[0].plot(weather.index, weather["TMAX"], color="tab:red", label="Daily high")
axes[0].plot(weather.index, weather["TMIN"], color="tab:blue", label="Daily low")

axes[0].set_ylabel("Temperature (F)")
axes[0].set_title("Nashville, 2025")
axes[0].legend()

# bottom: precipitation, as bars
axes[1].bar(weather.index, weather["PRCP"], color="tab:cyan")
axes[1].set_ylabel("Precipitation (in)")
axes[1].set_xlabel("Date")

Two things to notice:
- **Precipitation is drawn with `ax.bar()`**, not `ax.plot()`. Each day's rain is a separate amount, so a bar for each day makes more sense than a line connecting one day's rain to the next.
- **The two axes don't quite match**. The bars stick out a little past the line plot. If hte reader is supposed to line the two up by date, that's a problem...

### Sharing an axis
We can solve this issue using `sharex=True` which makes every graph use the **same x-axis**. Their x-limits match, and zooming one zooms all of them. You also only need to label the bottom one:

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)                             # NEW

# top: temperature
axes[0].plot(weather.index, weather["TMAX"], color="tab:red", label="Daily high")
axes[0].plot(weather.index, weather["TMIN"], color="tab:blue", label="Daily low")

axes[0].set_ylabel("Temperature (F)")
axes[0].set_title("Nashville, 2025")
axes[0].legend()

# bottom: precipitation, as bars
axes[1].bar(weather.index, weather["PRCP"], color="tab:cyan")
axes[1].set_xlim(pd.Timestamp("2025-01-01"), pd.Timestamp("2025-12-31"))                # NEW
axes[1].set_ylabel("Precipitation (in)")
axes[1].set_xlabel("Date")

We only called `set_xlim()` on `axes[1]`, but both graphs changed because they share one x-axis.

Now you can ask questions **across** two panels. The tallest bar is the year's wettest day: *8April 3, 3.86 in**, and another 2.12 in fell two days later. look straight up: the highs stay in the 80s through april 5, then drop into the 50s and 60s by April 6-8.

### Grids of graphs

With more than one row **and** more than one column, `axes` is a **2D** array. You can index ith with `[row, col]`, exactly like any other array:

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(9, 6))
print(axes.shape)

axes[0, 0].set_title("axes[0, 0]")
axes[0, 1].set_title("axes[0, 1]")
axes[1, 0].set_title("axes[1, 0]")
axes[1, 1].set_title("axes[1, 1]")

fig.tight_layout()

<div class="alert alert-success">

### **Tip**: `fig.tight_layout()`
With several graphs on one page, titles and tick labels can bump into each other. `fig.tight_layout()` nudges everything apart so nothing overlaps. Put it at the end of the cell, after all the labels are set. (It's somethin gyou do to the whole **page**, so it goes on `fig`)
    
</div>

<div class="alert alert-info">

### Exercise 1 : temperature, rain, and wind

Make a figure with **3 rows**, all sharing one x-axis, that is 10 inches wide and 8 inches tall. Plot:
- top: daily highs and lows, with a legend
- middle: daily precipitation as bars
- bottom: average daily wind speed, `AWND` as a line

Label every y-axis with units, put a title on the **top** graph only, label the x-axis on the **bottom** graph only, and finish with `fig.tight_layout()`.

**question**: the windiest day of the year was **March 4** (19.7 mph). Zoom all three panels to **March 1-10**. What are the temperature and rain doing around it?
    
</div>

In [ ]:
# your answer here
fig, axes = plt.subplots(___________________________)

axes[0].plot(_______, __________, color="tab:red", label="Daily high")
axes[0].plot(_______, __________, color="tab:blue", label="Daily low")
axes[0].set_ylabel("Temperature (F)")
axes[0].set_title("Nashville, early March 2025")
axes[0].legend()

axes[1].bar(_______, __________, color="tab:cyan")
axes[1].set_ylabel("Precipitation (in)")

axes[2].plot(_______, __________, color="tab:gray")
axes[2].set_ylabel("Wind speed (mph)")
axes[2].set_xlabel("Date")

axes[2].set_xlim(_______, __________)
fig.autofmt_xdate()
fig.tight_layout()

---
## Part 2 : bar charts, amounts in categories

A bar's **length** shows an amount. Bars work best when the x-axis is a set of **categories** (months, gullies, stations) rather than a smooth number line.

Monthly rain totals are a perfect fit, and you already know how to compute them with `groupby`:

In [ ]:
monthly_prcp = weather.groupby(weather.index.month)["PRCP"].sum()
monthly_prcp

The index is the month **numbers** 1-12. For a reader, month **names** are nicer, so we'll pass a list of names as the x values. `ax.bar(x, heights)` draws one bar per name:

In [ ]:
months = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

fig, ax = plt.subplots(figsize=(8, 4))

ax.bar(months, monthly_prcp, color="tab:cyan", edgecolor="black")

ax.set_ylabel("Total precipitation (in)")
ax.set_title("Nashville monthly precipitation, 2025");

- The first argument is the **categories** (12 names), the second is the **heights** (12 numbers). They line up by position, like `zip()`.
- Bars always start at **zero**, and that's on purpose: the reader compares bar **lengths**, and a bar that doesn't start at zero lies about its length. matplotlib does this for you; don't undo it with `set_ylim()`.
- `ax.barh()` makes **horizontal bars**, handy when category names are long.

You can see that April and May were the wettest months in 2025, with about 8.5 inches each.

<div class="alert alert-info">

### Exercise 2 : how much, and how often?

April and May had almost the same total rain. Did it fall the same way?
1. Make a Series `rainy_days`: the number of days in each month with **any** precipitation.
    - **hint**: make a mask, then group things, then sum it.
2. Make a figure with **1 row and 2 columns**, 11 inches wide and 4 inches tall. Label everything and finish with `fig.tight_layout()`
    - left: monthly total precipitation as bars
    - right: rainy days per month as bars.

**Question**: compare April and May. What does that tell you about **how** the rain fell each month?


</div>

In [ ]:
# your answer here
# 1
rainy = _______________ # make a mask
rainy_days = _______________ # use the mask

# 2
fig, axes = plt.subplots(_______________)

_______________ # make a bar plot
_______________.set_ylabel("Total precipitation (in)")
_______________.set_title("How much rain")

_______________ # make a bar plot
_______________.set_ylabel("Days with precipitation")
_______________.set_title("How often it rained")

fig.tight_layout()

---
## Part 3 : log axes, when values span powers of ten

In Lab 06, you wrote a function for the energy an earthquake releases:
$$E = 10^{\,1.5M + 4.8} \text{ joules}$$
*(the Guttenberg-Richter energy-magntiude relation, Gutenberg & Richter, 1956)*

Let's plot it for magnitudes 3 through 9, on two axes side by side. The left one is ordinary, the right one gets **one** extra line:

In [ ]:
def eq_energy(magnitude):
    """Seismic energy (J) releasted by an earthquake of a given magnitude"""
    x = 10 **(1.5 * magnitude + 1.8)
    return x

In [ ]:
mags = np.arange(3, 10)
energy = eq_energy(mags)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].plot(mags, energy, "o-")
axes[0].set_title("Linear y-axis")

axes[1].plot(mags, energy, "o-")
axes[1].set_yscale("log")            # New
axes[1].set_title("Log y-axis")

for ax in axes:
    ax.set_xlabel("Magnitude")
    ax.set_ylabel("Energy released (J)")

fig.tight_layout()

**Left**: M3 through M7 are all squazhed flat against zero, and only M8 and M9 show up. Each step in magnitude multiplies the energy by about 32, so the biggest value swamps the rest.

**Right**: on a **log** axis, each tick is **10 times** the one below it (10$^{10}$, 10$^{12}$, etc). Equal **ratios** get equal **spacing**, so "x32 every step" becomes a straight line, and every magnitude is visible.

Use a log axis when your values cover **several powers of ten**, or when you carea bout **how many times bigger** rather than **how much bigger**.

|**you want**|**write**|
|:--|:--|
|log y-axis| `ax.set_yscale("log")`|
|log x-axis| `ax.set_xscale("log")`|
|both| both lines, or draw with `ax.loglog(x, y)` isntead of `ax.plot(x, y)`|

Also notice the `for ax in axes:` loop: when every graph needs the same labels, loop over the axes instead of writing the same lines twice.

### common mistake: zeros on a log axis

The log of 0 doesn't exist. So a value of **0** can't be drawn on a log axis. matplotlib quietly leaves it out. If your data has zeros (like dry days in `PRCP`), decide on purpose what to do with them, for example, by plotting only the rainy days.

In [ ]:
rain = weather.loc[weather["PRCP"] > 0, "PRCP"]

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

for ax in axes:
    ax.hist(rain, bins=12, color="tab:cyan", edgecolor="black")
    ax.set_xlabel("Daily precipitation (in)")
    ax.set_ylabel("Number of days")

axes[0].set_title("Linear y-axis")
axes[1].set_yscale("log")
axes[1].set_title("Log y-axis")

fig.tight_layout()

On the linear axis, the first bar (the smallest amounts) towers over everything, and the bars for the big storms are so short they're nearly invisible. On the log axis, you can read **every** var, including hte single days near 2, 3, and 4 inches. Lots of small rainy days and a few big ones is the normal pattern for daily rainfall.

---
## Part 4 : what makes a good figure?

A figure is **finished** when it runs. It's **good** when someone else can read it without you in the room. Here's a checklist to use for the rest of the semester:

|**#**|**check**|**the question to ask**|
|:--|:--|:--|
|1| **labels with units**|Does every axis say what it is **and** its units?|
|2| **a title that says what, where, when**|Could someone tell what this is without the code?|
|3| **a legend** for more than one line|Can the reader tell which line is which?|
|4| **one kind of unit per y-axis**|Are F and inches sharing an axis? Use subplots instead|
|5| **colors anyone can tell apart**|Does it only work if you can tell red from green? Also vary `linestyle` or `marker`|
|6| **readable size**|Can you read the tick labels without zooming?|
|7| **compare like with like**|If panels are meant to be compared, do they share axes? Do bars start at zero?|

### Making text bigger
`fontsize=` works on any label or title, and `ax.tick_params(labelsize=...)` sets the numbers along the axes:
```python
ax.set_xlabel("Month", fontsize=13)
ax.set_title("...", fontsize=14)
ax.tick_params(labelsize=11)
```
### Here's a figure that breaks the rules


In [ ]:
monthly_high = weather.groupby(weather.index.month)["TMAX"].mean()
monthly_low = weather.groupby(weather.index.month)["TMIN"].mean()

fig, ax = plt.subplots(figsize=(4, 2.5))
ax.plot(monthly_high, color="red")
ax.plot(monthly_low, color="green")
ax.plot(monthly_prcp, color="lime")

<div class="alert alert-info">

### Exercise 3 : fix it

Remake the figure above so it passes every check. You'll need **two** graphs (side by side or stacked, your choice). Use month names on the x-axis, and show precipitation as **bars**. Make the text big enough to read
    
</div>

In [ ]:
# your answer here

---
## A useful table

| to... | write |
|---|---|
| several graphs on one page | `fig, axes = plt.subplots(rows, cols, figsize=(w, h))` |
| pick one graph | `axes[0]` (one row or column) or `axes[0, 1]` (a grid) |
| line panels up | `sharex=True`, `sharey=True` |
| keep labels from overlapping | `fig.tight_layout()` |
| one title for the whole page | `fig.suptitle("...")` |
| bars for amounts in categories | `ax.bar(categories, heights)`, `ax.barh(...)` |
| log scale | `ax.set_yscale("log")`, `ax.set_xscale("log")`, `ax.loglog(x, y)` |
| bigger text | `fontsize=` on labels, `ax.tick_params(labelsize=...)` |
| same labels on every graph | `for ax in axes:` (for a grid, `for ax in axes.flat:`) |

### References
- Gutenberg, B., & Richter, C. F. (1956). Magnitude and energy of earthquakes. *Annali di Geofisica*, 9, 1–15.
- Jake VanderPlas, [*Python Data Science Handbook*, Chapter 4](https://jakevdp.github.io/PythonDataScienceHandbook/04.00-introduction-to-matplotlib.html). Section 4.08, "Multiple Subplots," matches Part 1.
- [matplotlib gallery](https://matplotlib.org/stable/gallery/index.html)